# MeChess 7/7: score Maia-3 on the test sets and write the human-likeness reports (GPU notebook)

An adapter around `chessme humanlike-score` and `chessme humanlike-report`. It downloads the Maia-3 code and the 5M / 23M / 79M weights yourself (AGPL-3.0: nothing is redistributed; they stay in this session), scores each model on both test sets
of notebook 6, and writes a report per test set: top-1, **sampled match** (the chance a move sampled from the model equals the human's), negative log-likelihood, calibration and **over- or under-confidence by the mover's clock**, by rating and phase, and the paired differences between models.

**Before you run:** add notebook 6's output as an input (*Add Input -> Notebook output files*); set *Accelerator -> GPU* and *Internet -> On*; **Save Version -> Save & Run All (Commit)**.
Scoring is limited by encoding boards in Python, so four worker processes share the GPU.

In [ ]:
import glob, os, shutil, subprocess, sys

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here (not needed when the repository is attached as a dataset)
SIZES = ["5m", "23m", "79m"]
WORKERS = 4
OUT = "/kaggle/working/humanlike" if os.path.exists("/kaggle") else "humanlike_local"
TESTSET_DIR = ""                                        # notebook 6's output folder; empty: search /kaggle/input
MAIA3_COMMIT = ""                                       # empty: the current default branch; a commit hash pins the baseline exactly

def sh(*args):
    """Run a command and stream its output; stop the notebook if it fails."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(args)}")

if not os.path.exists("chessme"):
    attached = glob.glob("/kaggle/input/**/chessme/__init__.py", recursive=True)     # the repository attached as a dataset
    if attached:
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), "MeChess")
    else:                                                           # Kaggle: fetch the repository (Internet must be on)
        if "<you>" in REPO_URL:
            raise SystemExit("Attach the repository as a dataset, or set REPO_URL.")
        sh("git", "clone", "--depth", "1", REPO_URL, "MeChess")
    os.chdir("MeChess")
sh(sys.executable, "-m", "pip", "-q", "install", "python-chess", "numpy", "pyyaml", "requests", "zstandard", "huggingface-hub")
CLI = [sys.executable, "-m", "chessme"]              # every step below is one `chessme` command: the same ones you run on a laptop

In [ ]:
found = sorted(glob.glob((TESTSET_DIR or "/kaggle/input") + "/**/testset_T[AB].jsonl.gz", recursive=True))
print("test sets found:", found)
TB = next((f for f in found if f.endswith("testset_TB.jsonl.gz")), None)
TA = next((f for f in found if f.endswith("testset_TA.jsonl.gz")), None)
if not TB:
    raise SystemExit("Add notebook 6's output as an input (testset_TB.jsonl.gz; testset_TA.jsonl.gz is scored too when present).")
os.makedirs(OUT, exist_ok=True)

## Step 1. The Maia-3 code and weights (external, not part of this repository)

In [ ]:
sh("git", "clone", "--depth", "1", "https://github.com/CSSLab/maia3.git", "/tmp/maia3")
if MAIA3_COMMIT:
    sh("git", "-C", "/tmp/maia3", "fetch", "--depth", "1", "origin", MAIA3_COMMIT)
    sh("git", "-C", "/tmp/maia3", "checkout", MAIA3_COMMIT)
sh("git", "-C", "/tmp/maia3", "log", "-1", "--format=maia3 commit %h %ad")
os.makedirs("/tmp/weights", exist_ok=True)
for size in SIZES:
    sh("curl", "-L", "-s", "-o", f"/tmp/weights/maia3-{size}.pt", f"https://huggingface.co/UofTCSSLab/Maia3-{size.upper()}/resolve/main/maia3-{size}.pt")
sh("sha256sum", *[f"/tmp/weights/maia3-{size}.pt" for size in SIZES])

## Step 2. Score every model on both test sets (history on, both ratings as input)

In [ ]:
for name, testset in [(k, v) for k, v in (("TA", TA), ("TB", TB)) if v]:
    for size in SIZES:
        sh(*CLI, "humanlike-score", "--testset", testset, "--scorer", f"maia3={size}+history", "--maia3-repo", "/tmp/maia3", "--weights-dir", "/tmp/weights",
           "--device", "cuda", "--workers", str(WORKERS), "--out", f"{OUT}/scores_{name}_{size}.npz")

## Step 3. Reports: one per test set, with the paired difference of the first two models

In [ ]:
for name, testset in [(k, v) for k, v in (("TA", TA), ("TB", TB)) if v]:
    sh(*CLI, "humanlike-report", "--testset", testset, "--scores", *[f"{OUT}/scores_{name}_{size}.npz" for size in SIZES], "--boot", "1000", "--out", f"{OUT}/report_{name}.md")